In [ ]:
# @title
!pip install bert-score
!pip install datasets
!pip install sentence-transformers datasets accelerate transformers wandb
!pip install sentence-transformers torch sklearn bert-score
!pip install sentencepiece
!pip install -q sentencepiece transformers sentence-transformers bert-score torch scikit-learn pandas numpy
!pip install -q pytorch-crf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


Fine tuned Post-Segmentation Normalization (FEMSeg_kaz+MAPP)


In [ ]:
# @title
# ============================================================
# FULL PIPELINE: Fine-tuning (5 epoch) + FEMSeg + MAPP + Hybrid Retrieval
#                + Lite Reranking + Answer Post-processing
#
# СХЕМА:
#   1) Деректер жүктеу + train/test бөлу
#   2) Fine-tuning (MultipleNegativesRankingLoss, 5 epoch)
#      - Fine-tuned модель сақталады -> SAVE_DIR
#   3) FEMSeg_v3 (Morphological Segmentation)
#   4) MAPP (Morphology-Aware Normalization)
#   5) Hybrid Retrieval: Dense (Fine-tuned MiniLM) + Morphological Similarity
#   6) Lite Reranking
#   7) Answer Post-processing
#   8) Eval: Exact@1, TokenF1@1, MeanCos@1(QSim),
#            Semantic@1(ans_cos>=0.85), BERTScoreF1@1
#   9) CSV экспорт
#
# Орнату (Colab-та бір рет орындаңыз):
#   !pip install -q sentence-transformers scikit-learn torchcrf
#   !pip install -q bert-score   # қажет болса
# ============================================================

# =========================
# ENV
# =========================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_DISABLED"] = "true"

import re
import csv
import glob
import json
import time
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from bert_score import score as bert_score_fn
except Exception:
    bert_score_fn = None

try:
    from torchcrf import CRF
except Exception:
    CRF = None

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# --------------------------
# Safety checks
# --------------------------
if CRF is None:
    raise ImportError(
        "torchcrf табылмады. Colab-та мына команданы орындаңыз:\n"
        "!pip install torchcrf"
    )

# --------------------------
# Determinism
# --------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥 DEVICE:", DEVICE)


# ============================================================
# CONFIG
# ============================================================
DATA_PATH   = "50k.json"
MODEL_NAME  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SEED        = 42
TEST_SIZE   = 0.10
SEM_THR     = 0.85

ENCODE_BATCH_SIZE    = 64
SIM_BATCH_SIZE       = 256
PRINT_PROGRESS_EVERY = 5000

EXPORT_CSV = True
CSV_PATH   = "full_pipeline_test_details.csv"

# Fine-tune config
DO_FINETUNE        = True
FINETUNE_EPOCHS    = 5
FINETUNE_BATCH     = 32       # OOM болса: 16
FINETUNE_LR        = 2e-5
FINETUNE_WARMUP_FR = 0.10
FINETUNE_MAX_PAIRS = None     # None -> барлық train жұбы
SAVE_DIR           = "finetuned_fullpipeline_minilm_5ep"

# Hybrid retrieval weights
ALPHA_SEG          = 0.40
HYBRID_DENSE_WEIGHT = 0.72
HYBRID_MORPH_WEIGHT = 0.28
DENSE_TOPK         = 80

# Reranking weights
RERANK_TOPK         = 12
RERANK_HYBRID_WEIGHT = 0.75
RERANK_ANS_WEIGHT   = 0.25

# MAPP config
USE_MAPP         = True
MAPP_TAG         = "[MAPP]"
MAPP_MIN_WORD_LEN = 4

POSTPROCESS_FOR_EVAL = True


# ============================================================
# 0) Robust JSON loading
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p
    for c in [f"/content/{p}", f"/content/drive/MyDrive/{p}"]:
        if Path(c).exists():
            return c
    hits = glob.glob(f"**/{Path(p).name}", recursive=True)
    if hits:
        return hits[0]
    near = glob.glob("**/*.json", recursive=True)
    raise FileNotFoundError(
        f"❌ File not found: {p}\nPWD: {Path.cwd()}\n"
        f"Found .json (first 30):\n" + "\n".join(near[:30])
    )

def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)

def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)

def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text

def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None
    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    return None

def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)
    if normalized is not None:
        return [normalized]
    if isinstance(obj, list):
        return [n for item in obj for n in [_normalize_record(item)] if n]
    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                return [n for item in obj[key] for n in [_normalize_record(item)] if n]
    return []

def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    records = _extract_records(json.loads(_normalize_json_text(text)))
    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")
    return records

def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx, records = 0, []
    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1
        if idx >= len(text):
            break
        obj, end = decoder.raw_decode(text, idx)
        idx = end
        records.extend(_extract_records(obj))
    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")
    return records

def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records, bad_lines = [], []
    for line_no, line in enumerate(text.splitlines(), start=1):
        s = _normalize_json_text(line.strip().rstrip(","))
        if not s:
            continue
        try:
            records.extend(_extract_records(json.loads(s)))
        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))
    if not records:
        preview = "\n".join([f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]])
        raise ValueError("Файлдан бірде-бір QA жазба оқылмады.\n" + preview)
    if bad_lines:
        print(f"⚠ {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")
    return records

def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []
    for parser in (_parse_as_single_json, _parse_as_multiple_json_objects, _parse_line_by_line):
        try:
            records = parser(text)
            if records:
                return records
        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")
    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


# ============================================================
# 1) Load data
# ============================================================

data_path = find_data_path(DATA_PATH)
print(f"📥 QA JSON жүктеу: {data_path}")

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON-да question/answer немесе instruction/response болуы тиіс."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"]   = qa_data["answer"].astype(str).str.strip()
qa_data = qa_data[(qa_data["question"] != "") & (qa_data["answer"] != "")].reset_index(drop=True)

print(f"📚 Жалпы QA жазба саны: {len(qa_data)}")
print(qa_data.head(3))


# ============================================================
# 2) Base text normalization
# ============================================================

_punct_space_left  = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space       = re.compile(r"\s+")

def clean_view(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t

def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_view(text).lower()).strip()

def tokens(text: str) -> List[str]:
    return re.findall(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+", clean_view(text).lower())

def token_f1(pred: str, gold: str) -> float:
    p, g = tokens(pred), tokens(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    inter = sum((pc & gc).values())
    if inter == 0:
        return 0.0
    prec = inter / max(1, len(p))
    rec  = inter / max(1, len(g))
    return (2 * prec * rec) / (prec + rec + 1e-12)

def cosine01(x: float) -> float:
    return max(0.0, min(1.0, (x + 1.0) * 0.5))


# ============================================================
# 3) FEMSeg_v3 (Morphological Segmentation)
# ============================================================

BMES_TAGS = ["B", "M", "E", "S"]
TAG2ID = {t: i for i, t in enumerate(BMES_TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}

class FEMSegV3(nn.Module):
    def __init__(self, vocab_size: int, char_emb_dim: int = 128,
                 lstm_hidden_dim: int = 256, dropout: float = 0.3,
                 num_tags: int = len(BMES_TAGS)):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(char_emb_dim, lstm_hidden_dim, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.fc      = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf     = CRF(num_tags, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward_features(self, x, mask):
        emb = self.dropout(self.emb(x))
        h, _ = self.lstm(emb)
        return self.dropout(h)

    def forward_logits(self, x, mask):
        return self.fc(self.forward_features(x, mask))

    def forward(self, x, tags=None, mask=None):
        if mask is None:
            mask = x != 0
        emissions = self.forward_logits(x, mask)
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)


def bmes_to_cse(chars: List[str], tags: List[str]) -> str:
    morphs, cur = [], ""
    for ch, t in zip(chars, tags):
        if t == "B":
            if cur: morphs.append(cur)
            cur = ch
        elif t == "M":
            cur += ch
        elif t == "E":
            morphs.append(cur + ch); cur = ""
        elif t == "S":
            if cur: morphs.append(cur)
            morphs.append(ch); cur = ""
    if cur:
        morphs.append(cur)
    return "@@ ".join(morphs)


class KazMorphSegmentor:
    def __init__(self, char2id_path: str, ckpt_path: str, use_cuda: bool = True):
        with open(char2id_path, "r", encoding="utf-8") as f:
            self.char2id: Dict[str, int] = json.load(f)
        self.pad_id = 0
        self.unk_id = self.char2id.get("<unk>") or self.char2id.get("[UNK]") or 1
        self.device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")

        vocab_size = max(self.char2id.values()) + 1
        self.model = FEMSegV3(vocab_size=vocab_size)
        self.model.to(self.device)

        state = torch.load(ckpt_path, map_location=self.device)
        if isinstance(state, dict):
            state = state.get("state_dict") or state.get("model_state_dict") or state
        if isinstance(state, dict) and any(k.startswith("module.") for k in state):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}
        self.model.load_state_dict(state, strict=True)
        self.model.eval()

        self.word_cache: Dict[str, str] = {}
        self.text_cache: Dict[str, str] = {}

    def _word_to_tensor(self, word: str):
        ids = [self.char2id.get(ch, self.unk_id) for ch in list(word)]
        x = torch.tensor([ids], dtype=torch.long, device=self.device)
        return x, (x != self.pad_id)

    def segment_word(self, word: str) -> str:
        word = word.strip()
        if not word: return ""
        if word in self.word_cache: return self.word_cache[word]
        x, mask = self._word_to_tensor(word)
        with torch.no_grad():
            best_paths = self.model(x, mask=mask)
        result = bmes_to_cse(list(word), [ID2TAG[i] for i in best_paths[0]])
        self.word_cache[word] = result
        return result

    def segment_text(self, text: str) -> str:
        text = text.strip()
        if not text: return ""
        if text in self.text_cache: return self.text_cache[text]
        result = " | ".join(self.segment_word(w) for w in text.split())
        self.text_cache[text] = result
        return result


def femseg_to_sp(text: str) -> str:
    text = clean_view(str(text))
    if not text: return ""
    seg_line = femseg.segment_text(text)
    sp_tokens = []
    for word_seg in seg_line.split(" | "):
        morphs = [m.strip() for m in word_seg.strip().split("@@ ") if m.strip()]
        if not morphs: continue
        sp_tokens.append("▁" + morphs[0])
        sp_tokens.extend(morphs[1:])
    return " ".join(sp_tokens)

def sp_token_set(sp_text: str) -> Set[str]:
    return {tok.replace("▁", "").strip() for tok in str(sp_text).split() if tok.replace("▁", "").strip()}


# ============================================================
# FEMSeg жолдары — Google Drive-тен
# ============================================================
if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)

FEMSEG_CHAR2ID = "/content/drive/MyDrive/KAZ_MORPH/char2id_femseg_v3_50k.json"
FEMSEG_CKPT    = "/content/drive/MyDrive/KAZ_MORPH/femseg_v3_50k_epoch3.pt"

assert os.path.exists(FEMSEG_CHAR2ID), f"char2id файлы табылмады: {FEMSEG_CHAR2ID}"
assert os.path.exists(FEMSEG_CKPT),    f"checkpoint файлы табылмады: {FEMSEG_CKPT}"

print("⬇️ FEMSeg_v3 жүктелуде...")
femseg = KazMorphSegmentor(FEMSEG_CHAR2ID, FEMSEG_CKPT, use_cuda=True)
print("✅ FEMSeg_v3 дайын.")
print("🔎 FEMSeg тест:", femseg_to_sp("Мен мектепке барамын"))


# ============================================================
# 4) MAPP — Morphology-Aware Normalization
# ============================================================

WORD_RE = re.compile(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9_+#-]+", re.UNICODE)

PROTECTED_WORDS = {
    "не", "неге", "неліктен", "қалай", "қайда", "қайдан", "қашан",
    "қандай", "қанша", "қай", "кім", "кімге", "кімнің", "неше",
    "python", "def", "for", "while", "if", "elif", "else", "print", "input",
    "list", "dict", "tuple", "set", "int", "str", "float", "bool", "none",
    "true", "false", "class", "return", "break", "continue", "range", "len",
    "append", "insert", "remove", "pop", "sort", "lambda", "map", "filter",
    "reduce", "zip", "enumerate", "and", "or", "not", "in", "is"
}

MAPP_WORD_MAP = {
    "калай": "қалай", "кандай": "қандай", "кайда": "қайда",
    "кашан": "қашан", "канша": "қанша", "аныкта": "анықта",
    "аныктайды": "анықтайды", "аныктаймыз": "анықтаймыз",
    "есептеймз": "есептейміз", "пайтон": "python", "питон": "python",
    "функсия": "функция", "функсиа": "функция", "циклд": "цикл",
    "стр": "str", "инт": "int", "бул": "bool", "принт": "print", "инпут": "input",
}

_mapp_word_cache: Dict[str, str] = {}
_mapp_text_cache: Dict[str, str] = {}

def _basic_mapp_word_norm(word: str) -> str:
    return MAPP_WORD_MAP.get(str(word).strip().lower(), str(word).strip().lower())

def _is_ascii_programming_token(word: str) -> bool:
    return bool(re.fullmatch(r"[a-z0-9_+#-]+", word))

def _mapp_stem_from_femseg(word: str) -> str:
    w = _basic_mapp_word_norm(word)
    if not USE_MAPP or not w: return w
    if w in _mapp_word_cache: return _mapp_word_cache[w]
    if w in PROTECTED_WORDS or len(w) < MAPP_MIN_WORD_LEN or w.isdigit() or _is_ascii_programming_token(w):
        _mapp_word_cache[w] = w; return w
    try:
        morphs = [m.strip() for m in femseg.segment_word(w).split("@@ ") if m.strip()]
        stem = _basic_mapp_word_norm(morphs[0]) if morphs else w
        result = stem if len(stem) >= 2 else w
    except Exception:
        result = w
    _mapp_word_cache[w] = result
    return result

def mapp_normalize(text: str) -> str:
    t = clean_view(text).lower().strip()
    if not t: return ""
    if t in _mapp_text_cache: return _mapp_text_cache[t]
    out, prev = [], None
    for w in WORD_RE.findall(t):
        nw = _mapp_stem_from_femseg(w)
        if nw and nw != prev:
            out.append(nw); prev = nw
    result = " ".join(out).strip() or t
    _mapp_text_cache[t] = result
    return result

def mapp_token_set(text: str) -> Set[str]:
    return set(WORD_RE.findall(mapp_normalize(text)))

def compose_dense_view(q_clean: str, q_mapp: str) -> str:
    q_clean = clean_view(q_clean)
    q_mapp  = clean_view(q_mapp)
    if not q_mapp or norm_for_exact(q_clean) == norm_for_exact(q_mapp):
        return q_clean
    return f"{q_clean} {MAPP_TAG} {q_mapp}"


# ============================================================
# 5) Answer Post-processing
# ============================================================

POST_WORD_MAP = {
    "пайтон": "Python", "питон": "Python",
    "функсиа": "функция", "функсия": "функция",
    "принт": "print", "инпут": "input",
}

def _is_code_like_line(line: str) -> bool:
    s = line.rstrip()
    if not s.strip(): return False
    if s.startswith("    ") or s.startswith("\t"): return True
    code_heads = ("def ", "for ", "while ", "if ", "elif ", "else:", "return ", "class ", "print(")
    if s.lstrip().startswith(code_heads): return True
    if re.match(r"^\s*#.*$", s): return True
    return False

def _replace_whole_words(text: str, mapping: Dict[str, str]) -> str:
    out = text
    for a, b in mapping.items():
        out = re.sub(rf"\b{re.escape(a)}\b", b, out, flags=re.IGNORECASE)
    return out

def answer_postprocess(text: str) -> str:
    if text is None: return ""
    t = str(text).replace("\r\n", "\n").strip()
    if not t: return ""
    out_lines, prev_norm = [], None
    for line in t.split("\n"):
        raw = line.rstrip()
        if not raw.strip():
            if out_lines and out_lines[-1] != "":
                out_lines.append("")
            prev_norm = None
            continue
        cleaned = raw.rstrip() if _is_code_like_line(raw) else _replace_whole_words(clean_view(raw), POST_WORD_MAP)
        normed = cleaned.strip()
        if normed != prev_norm:
            out_lines.append(cleaned)
            prev_norm = normed
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out_lines).strip()).strip()


# ============================================================
# 6) Train / Test split
# ============================================================

train_df, test_df = train_test_split(
    qa_data, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)
print(f"\n📊 TRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED} | test_size={TEST_SIZE}")

train_rows = train_df[["question", "answer"]].to_dict("records")
test_rows  = test_df[["question", "answer"]].to_dict("records")


# ============================================================
# 7) Fine-tuning (MultipleNegativesRankingLoss, 5 epoch)
# ============================================================

def finetune_model(base_model_name: str, rows: List[Dict], save_dir: str) -> SentenceTransformer:
    from torch.utils.data import DataLoader
    from sentence_transformers import InputExample, losses

    if Path(save_dir).exists() and any(Path(save_dir).glob("*")):
        print(f"\n✅ Fine-tuned модель бар: {save_dir} -> жүктелуде")
        return SentenceTransformer(save_dir, device=DEVICE)

    print("\n==================== FINE-TUNING ====================")
    print(f"Base model  : {base_model_name}")
    print(f"Epochs      : {FINETUNE_EPOCHS} | Batch: {FINETUNE_BATCH} | LR: {FINETUNE_LR}")

    model = SentenceTransformer(base_model_name, device=DEVICE)

    samples = [
        InputExample(texts=[clean_view(r["question"]), clean_view(r["answer"])])
        for r in rows if clean_view(r["question"]) and clean_view(r["answer"])
    ]
    if FINETUNE_MAX_PAIRS is not None:
        samples = samples[:int(FINETUNE_MAX_PAIRS)]
    if len(samples) < 2:
        raise ValueError("❌ Fine-tuning үшін жеткілікті train жұбы жоқ.")

    loader = DataLoader(samples, shuffle=True, batch_size=FINETUNE_BATCH, drop_last=True)
    loss   = losses.MultipleNegativesRankingLoss(model)
    warmup = int(len(loader) * FINETUNE_EPOCHS * FINETUNE_WARMUP_FR)

    print(f"Train pairs : {len(samples)} | Steps/epoch: {len(loader)} | Warmup: {warmup}")

    model.fit(
        train_objectives=[(loader, loss)],
        epochs=FINETUNE_EPOCHS,
        warmup_steps=warmup,
        optimizer_params={"lr": FINETUNE_LR},
        show_progress_bar=True,
        use_amp=torch.cuda.is_available(),
    )

    Path(save_dir).mkdir(parents=True, exist_ok=True)
    model.save(save_dir)
    print(f"✅ Fine-tuned модель сақталды: {save_dir}")
    return SentenceTransformer(save_dir, device=DEVICE)


if DO_FINETUNE:
    retr_model = finetune_model(MODEL_NAME, train_rows, SAVE_DIR)
    print(f"\n✅ Модель (fine-tuned): {SAVE_DIR}")
else:
    retr_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    print(f"\n✅ Модель (base): {MODEL_NAME}")

retr_model.eval()


# ============================================================
# 8) Build all views (MAPP + FEMSeg)
# ============================================================

qa_data["q_clean"] = qa_data["question"].map(clean_view)
qa_data["a_clean"] = qa_data["answer"].map(clean_view)

print(f"\n🧠 MAPP нормализация... ({len(qa_data)} жазба)")
t0 = time.time()
unique_qclean = qa_data["q_clean"].unique().tolist()
q_mapp_map = {}
for i, txt in enumerate(unique_qclean, 1):
    q_mapp_map[txt] = mapp_normalize(txt)
    if i % PRINT_PROGRESS_EVERY == 0 or i == len(unique_qclean):
        print(f"   MAPP {i}/{len(unique_qclean)} | {time.time()-t0:.1f} сек")

qa_data["q_mapp"]  = qa_data["q_clean"].map(q_mapp_map)
qa_data["q_dense"] = qa_data.apply(lambda r: compose_dense_view(r["q_clean"], r["q_mapp"]), axis=1)

print(f"\n🧩 FEMSeg сегментация... ({len(qa_data)} жазба)")
t0 = time.time()
unique_qmapp = qa_data["q_mapp"].unique().tolist()
q_sp_map = {}
for i, txt in enumerate(unique_qmapp, 1):
    q_sp_map[txt] = femseg_to_sp(txt)
    if i % PRINT_PROGRESS_EVERY == 0 or i == len(unique_qmapp):
        print(f"   SEG {i}/{len(unique_qmapp)} | {time.time()-t0:.1f} сек")

qa_data["q_sp"]    = qa_data["q_mapp"].map(q_sp_map)
qa_data["a_final"] = qa_data["a_clean"].map(answer_postprocess)

# Sync back train/test rows
train_idx = train_df.index.to_numpy()
test_idx  = test_df.index.to_numpy()

train_answers = [qa_data.loc[i, "a_final"] for i in train_idx]
test_answers  = [qa_data.loc[i, "a_final"] for i in test_idx]


# ============================================================
# 9) Embedding helpers
# ============================================================

def _encode_dual(dense_texts: List[str], seg_texts: List[str],
                 batch_size: int = ENCODE_BATCH_SIZE, normalize: bool = True) -> np.ndarray:
    v_seg = retr_model.encode(
        [str(t) for t in seg_texts], batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False)
    v_den = retr_model.encode(
        [str(t) for t in dense_texts], batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False)
    vecs = ALPHA_SEG * v_seg + (1.0 - ALPHA_SEG) * v_den
    if normalize:
        vecs /= (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
    return vecs.astype(np.float32)

def _encode_answers(texts: List[str], batch_size: int = ENCODE_BATCH_SIZE,
                    normalize: bool = True) -> np.ndarray:
    clean = [answer_postprocess(t) for t in texts]
    return retr_model.encode(
        clean, batch_size=batch_size, convert_to_numpy=True,
        normalize_embeddings=normalize, show_progress_bar=False).astype(np.float32)

def _build_query_views(text: str):
    q_clean = clean_view(text)
    q_mapp  = mapp_normalize(q_clean)
    q_dense = compose_dense_view(q_clean, q_mapp)
    q_sp    = femseg_to_sp(q_mapp)
    return q_clean, q_mapp, q_dense, q_sp, set(WORD_RE.findall(q_mapp)), sp_token_set(q_sp)


# ============================================================
# 10) Precompute full embeddings
# ============================================================

qa_q_dense_full = qa_data["q_dense"].tolist()
qa_q_sp_full    = qa_data["q_sp"].tolist()
qa_a_final_full = qa_data["a_final"].tolist()
qa_q_mapp_full  = qa_data["q_mapp"].tolist()

print("\n📦 FULL question embedding...")
t0 = time.time()
qa_q_emb_full = _encode_dual(qa_q_dense_full, qa_q_sp_full)
print(f"✅ {time.time()-t0:.2f} сек")

print("📦 FULL answer embedding...")
t0 = time.time()
qa_a_emb_full = _encode_answers(qa_a_final_full)
print(f"✅ {time.time()-t0:.2f} сек")

qa_q_tok_sets_full = [set(WORD_RE.findall(x)) for x in qa_q_mapp_full]
qa_q_seg_sets_full = [sp_token_set(x) for x in qa_q_sp_full]

# Split arrays
train_q_emb = qa_q_emb_full[train_idx]
test_q_emb  = qa_q_emb_full[test_idx]
train_a_emb = qa_a_emb_full[train_idx]
test_a_emb  = qa_a_emb_full[test_idx]

train_q_tok_sets = [qa_q_tok_sets_full[i] for i in train_idx]
test_q_tok_sets  = [qa_q_tok_sets_full[i] for i in test_idx]
train_q_seg_sets = [qa_q_seg_sets_full[i] for i in train_idx]
test_q_seg_sets  = [qa_q_seg_sets_full[i] for i in test_idx]


# ============================================================
# 11) Hybrid Retrieval helpers
# ============================================================

def jaccard_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return len(a & b) / max(1, len(a | b))

def dice_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return (2.0 * len(a & b)) / max(1.0, len(a) + len(b))

def morph_similarity(q_tok: Set[str], c_tok: Set[str], q_seg: Set[str], c_seg: Set[str]) -> float:
    return 0.65 * jaccard_sim(q_tok, c_tok) + 0.35 * dice_sim(q_seg, c_seg)

def get_topk_sorted(scores: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[0])
    if k <= 0: return np.array([], dtype=np.int32), np.array([], dtype=np.float32)
    idx = np.argpartition(-scores, k - 1)[:k]
    vals = scores[idx]
    order = np.argsort(-vals)
    return idx[order].astype(np.int32), vals[order].astype(np.float32)


# ============================================================
# 12) Lite Reranking
# ============================================================

def rerank_candidates_lite(q_emb, q_tok, q_seg, cand_idx, cand_scores,
                           kb_q_tok, kb_q_seg, kb_a_emb):
    rows = []
    for idx, dscore in zip(cand_idx, cand_scores):
        morph = morph_similarity(q_tok, kb_q_tok[idx], q_seg, kb_q_seg[idx])
        hybrid = HYBRID_DENSE_WEIGHT * cosine01(float(dscore)) + HYBRID_MORPH_WEIGHT * morph
        rows.append((idx, float(dscore), float(morph), float(hybrid)))

    rows.sort(key=lambda x: x[3], reverse=True)
    rows = rows[:min(RERANK_TOPK, len(rows))]

    best, qv = None, q_emb.reshape(-1)
    for idx, dscore, morph, hybrid in rows:
        ans_rel = cosine01(float(np.dot(qv, kb_a_emb[idx])))
        rerank  = RERANK_HYBRID_WEIGHT * hybrid + RERANK_ANS_WEIGHT * ans_rel
        row = {"idx": idx, "dense_qsim": dscore, "morph_sim": morph,
               "hybrid_score": hybrid, "ans_rel": ans_rel - 0.5, "rerank_score": rerank}
        if best is None or row["rerank_score"] > best["rerank_score"]:
            best = row
    return best


# ============================================================
# 13) Retrieval runtime / eval
# ============================================================

_query_emb_cache: Dict[str, np.ndarray] = {}

def _get_query_emb(q_dense: str, q_sp: str) -> np.ndarray:
    key = f"{q_dense}|||{q_sp}"
    if key not in _query_emb_cache:
        _query_emb_cache[key] = _encode_dual([q_dense], [q_sp], batch_size=1)
    return _query_emb_cache[key]

def _retrieve_from_kb(question, kb_q_emb, kb_a_emb, kb_answers, kb_q_tok, kb_q_seg, threshold=0.0):
    q_clean, q_mapp, q_dense, q_sp, q_tok_set, q_seg_set = _build_query_views(question)
    qv   = _get_query_emb(q_dense, q_sp)
    sims = (kb_q_emb @ qv.T).squeeze(1)
    top_idx, top_vals = get_topk_sorted(sims, DENSE_TOPK)

    best = rerank_candidates_lite(
        q_emb=qv.squeeze(0), q_tok=q_tok_set, q_seg=q_seg_set,
        cand_idx=top_idx.tolist(), cand_scores=top_vals.tolist(),
        kb_q_tok=kb_q_tok, kb_q_seg=kb_q_seg, kb_a_emb=kb_a_emb,
    )

    if best is None:
        return "Кешіріңіз, нақты жауап табылмады.", -1, 0.0, 0.0, 0.0, q_clean, q_mapp, q_dense

    idx = int(best["idx"])
    qsim = float(best["dense_qsim"])

    if qsim < threshold:
        return "Кешіріңіз, нақты жауап табылмады.", -1, qsim, best["morph_sim"], best["rerank_score"], q_clean, q_mapp, q_dense

    ans = answer_postprocess(kb_answers[idx]) if POSTPROCESS_FOR_EVAL else kb_answers[idx]
    return ans, idx, qsim, float(best["morph_sim"]), float(best["rerank_score"]), q_clean, q_mapp, q_dense

def ask_question_runtime(question: str, threshold: float = 0.0):
    return _retrieve_from_kb(
        question, qa_q_emb_full, qa_a_emb_full, qa_a_final_full,
        qa_q_tok_sets_full, qa_q_seg_sets_full, threshold)

def ask_question_eval(question: str):
    return _retrieve_from_kb(
        question, train_q_emb, train_a_emb, train_answers,
        train_q_tok_sets, train_q_seg_sets, threshold=0.0)


# ============================================================
# 14) BERTScore helper
# ============================================================

def _bert_lang_try(preds: List[str], golds: List[str]) -> Optional[float]:
    if bert_score_fn is None: return None
    for lang in ("kk", "tr", "en"):
        try:
            _, _, F1 = bert_score_fn(preds, golds, lang=lang, rescale_with_baseline=True)
            arr = F1.numpy() if hasattr(F1, "numpy") else np.array(F1)
            return float(np.mean(arr))
        except Exception:
            continue
    return None


# ============================================================
# 15) Eval cache + predictions
# ============================================================

_eval_cache: Dict[str, Any] = {}

def precompute_eval_predictions():
    if "pred_answers" in _eval_cache: return

    print("\n⚡ TEST → TRAIN pipeline retrieval есептелуде...")
    t0 = time.time()

    pred_idx, pred_qsims, pred_msims, pred_rscores, pred_ans = [], [], [], [], []
    n_test  = len(test_q_emb)
    topk    = min(DENSE_TOPK, train_q_emb.shape[0])

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end   = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]
        sims  = batch @ train_q_emb.T

        local_top = np.argpartition(-sims, topk - 1, axis=1)[:, :topk]
        row_ids   = np.arange(end - start)[:, None]
        local_vals = sims[row_ids, local_top]
        order     = np.argsort(-local_vals, axis=1)
        local_top  = np.take_along_axis(local_top, order, axis=1)
        local_vals = np.take_along_axis(local_vals, order, axis=1)

        for r in range(end - start):
            gi = start + r
            best = rerank_candidates_lite(
                q_emb=batch[r], q_tok=test_q_tok_sets[gi], q_seg=test_q_seg_sets[gi],
                cand_idx=local_top[r].tolist(), cand_scores=local_vals[r].tolist(),
                kb_q_tok=train_q_tok_sets, kb_q_seg=train_q_seg_sets, kb_a_emb=train_a_emb,
            )
            if best is None:
                pred_idx.append(-1); pred_qsims.append(0.0)
                pred_msims.append(0.0); pred_rscores.append(0.0)
                pred_ans.append("Кешіріңіз, нақты жауап табылмады.")
            else:
                i = int(best["idx"])
                ans = answer_postprocess(train_answers[i]) if POSTPROCESS_FOR_EVAL else train_answers[i]
                pred_idx.append(i); pred_qsims.append(float(best["dense_qsim"]))
                pred_msims.append(float(best["morph_sim"])); pred_rscores.append(float(best["rerank_score"]))
                pred_ans.append(ans)

    _eval_cache.update({
        "pred_indices": np.array(pred_idx, dtype=np.int32),
        "pred_q_sims":  np.array(pred_qsims, dtype=np.float32),
        "pred_morph_sims":   np.array(pred_msims, dtype=np.float32),
        "pred_rerank_scores": np.array(pred_rscores, dtype=np.float32),
        "pred_answers":  pred_ans,
        "true_answers":  test_answers,
        "test_questions": [clean_view(qa_data.loc[i, "question"]) for i in test_idx],
    })
    print(f"✅ TEST prediction дайын: {time.time()-t0:.2f} сек")

def _encode_answer_pairs():
    if "pred_eval_emb" in _eval_cache: return _eval_cache["pred_eval_emb"], _eval_cache["true_eval_emb"]
    precompute_eval_predictions()
    print("\n📐 Answer embedding...")
    pv = _encode_answers(_eval_cache["pred_answers"])
    tv = _encode_answers(_eval_cache["true_answers"])
    _eval_cache["pred_eval_emb"] = pv
    _eval_cache["true_eval_emb"] = tv
    return pv, tv


# ============================================================
# 16) Metrics
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()
    return float(np.mean([
        float(norm_for_exact(p) == norm_for_exact(t))
        for p, t in zip(_eval_cache["pred_answers"], _eval_cache["true_answers"])
    ]))

def evaluate_token_f1_at_1():
    precompute_eval_predictions()
    return float(np.mean([
        token_f1(p, t)
        for p, t in zip(_eval_cache["pred_answers"], _eval_cache["true_answers"])
    ]))

def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()
    return float(np.mean(_eval_cache["pred_q_sims"]))

def evaluate_semantic_at_1(threshold: float = SEM_THR):
    pv, tv = _encode_answer_pairs()
    ans_cos = np.sum(pv * tv, axis=1)
    _eval_cache["ans_cos"] = ans_cos
    return float(np.mean(ans_cos >= threshold))

def evaluate_bertscore_f1_at_1():
    if "bertscore_f1_mean" in _eval_cache: return _eval_cache["bertscore_f1_mean"]
    precompute_eval_predictions()
    print("\n📊 BERTScoreF1@1 есептелуде...")
    t0 = time.time()
    f1 = _bert_lang_try(_eval_cache["pred_answers"], _eval_cache["true_answers"])
    _eval_cache["bertscore_f1_mean"] = f1
    print(f"✅ BERTScoreF1@1 дайын: {time.time()-t0:.2f} сек")
    return f1

def build_eval_details() -> pd.DataFrame:
    if "details_df" in _eval_cache: return _eval_cache["details_df"]
    precompute_eval_predictions()
    if "ans_cos" not in _eval_cache:
        evaluate_semantic_at_1(SEM_THR)
    ans_cos = _eval_cache["ans_cos"]
    rows = []
    for q, gold, pred, qsim, msim, rscore, acos in zip(
        _eval_cache["test_questions"], _eval_cache["true_answers"],
        _eval_cache["pred_answers"], _eval_cache["pred_q_sims"],
        _eval_cache["pred_morph_sims"], _eval_cache["pred_rerank_scores"], ans_cos
    ):
        rows.append({
            "test_question": q, "gold_answer": gold, "pred_answer": pred,
            "QSim": float(qsim), "MorphSim": float(msim), "RerankScore": float(rscore),
            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "TokenF1": float(token_f1(pred, gold)),
            "AnsCos": float(acos), "SemHit": float(acos >= SEM_THR),
        })
    df = pd.DataFrame(rows)
    _eval_cache["details_df"] = df
    return df

def export_details_csv(path: str):
    df = build_eval_details()
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV exported: {path}  (rows={len(df)})")


# ============================================================
# 17) Full evaluation
# ============================================================

def evaluate_system_full():
    t0 = time.time()
    precompute_eval_predictions()

    exact    = evaluate_exact_at_1()
    tf1      = evaluate_token_f1_at_1()
    qsim_avg = evaluate_mean_cos_qsim_at_1()
    sem      = evaluate_semantic_at_1(SEM_THR)
    bscore   = evaluate_bertscore_f1_at_1()
    total    = time.time() - t0

    print("\n==================== RESULTS (Fine-tuned + Full Schema) ====================")
    print("Pipeline: Fine-tuning (5 ep) + FEMSeg + MAPP + Hybrid + Reranking + PostProcess")
    print(f"{'Exact@1':>30}: {exact:.6f}")
    print(f"{'TokenF1@1':>30}: {tf1:.6f}")
    print(f"{'MeanCos@1(QSim)':>30}: {qsim_avg:.6f}")
    print(f"{'Semantic@1(ans_cos>='+str(SEM_THR)+')':>30}: {sem:.6f}")
    if bscore is not None:
        print(f"{'BERTScoreF1@1':>30}: {bscore:.6f}")
    else:
        print(f"{'BERTScoreF1@1':>30}: N/A (bert-score орнатылмаған)")
    print(f"{'EvaluationTime':>30}: {total:.2f} сек")
    print(f"\n{'DO_FINETUNE':>30}: {DO_FINETUNE}")
    print(f"{'FINETUNE_EPOCHS':>30}: {FINETUNE_EPOCHS}")
    print(f"{'SAVE_DIR':>30}: {SAVE_DIR}")
    print(f"{'ALPHA_SEG':>30}: {ALPHA_SEG}")
    print(f"{'HYBRID_DENSE_WEIGHT':>30}: {HYBRID_DENSE_WEIGHT}")
    print(f"{'HYBRID_MORPH_WEIGHT':>30}: {HYBRID_MORPH_WEIGHT}")
    print(f"{'RERANK_HYBRID_WEIGHT':>30}: {RERANK_HYBRID_WEIGHT}")
    print(f"{'RERANK_ANS_WEIGHT':>30}: {RERANK_ANS_WEIGHT}")
    print(f"{'DENSE_TOPK':>30}: {DENSE_TOPK}")
    print(f"{'RERANK_TOPK':>30}: {RERANK_TOPK}")

    if EXPORT_CSV:
        export_details_csv(CSV_PATH)


# ============================================================
# 18) Interactive QA
# ============================================================

def interactive():
    print(f"\n==================== INTERACTIVE QA ====================")
    print("Fine-tuned + FEMSeg + MAPP + Hybrid + Reranking")
    print("Шығу үшін: exit немесе quit\n")
    while True:
        q = input("Сұрақ: ").strip()
        if not q: continue
        if q.lower() in {"exit", "quit", "q"}: break

        ans, idx, qsim, msim, rscore, q_clean, q_mapp, q_dense = ask_question_runtime(q)

        print(f"\n=== Жауап (qsim={qsim:.4f}, morph={msim:.4f}, rerank={rscore:.4f}) ===")
        print(ans)
        print(f"\n--- Query views ---")
        print(f"clean : {q_clean}")
        print(f"mapp  : {q_mapp}")
        print(f"dense : {q_dense}\n")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    try:
        interactive()
    except EOFError:
        pass

    evaluate_system_full()

🖥 DEVICE: cuda
📥 QA JSON жүктеу: 50k.json
⚠ 256 жол оқылмады, бірақ қалғандары жүктелді.
📚 Жалпы QA жазба саны: 50275
                                  question  \
0                     Python дегеніміз не?   
1           Python тілі қалай пайда болды?   
2  Python тілінің басты ерекшелігі қандай?   

                                              answer  
0  Python — қарапайым және оқуға оңай бағдарламал...  
1  Python тілін 1991 жылы Гвидо ван Россум жасаға...  
2  Python — оқуға жеңіл және интуитивті синтаксис...  
Mounted at /content/drive
⬇️ FEMSeg_v3 жүктелуде...
✅ FEMSeg_v3 дайын.
🔎 FEMSeg тест: ▁Мен ▁мектеп ке ▁бар а мын

📊 TRAIN: 45247 | TEST: 5028 | seed=42 | test_size=0.1

==================== FINE-TUNING ====================
Base model  : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Epochs      : 5 | Batch: 32 | LR: 2e-05


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScoreF1@1 дайын: 72.21 сек

==================== RESULTS (Fine-tuned + Full Schema) ====================
Pipeline: Fine-tuning (5 ep) + FEMSeg + MAPP + Hybrid + Reranking + PostProcess
                       Exact@1: 0.017104
                     TokenF1@1: 0.520100
               MeanCos@1(QSim): 0.846240
     Semantic@1(ans_cos>=0.85): 0.355012
                 BERTScoreF1@1: 0.850590
                EvaluationTime: 95.50 сек

                   DO_FINETUNE: True
               FINETUNE_EPOCHS: 5
                      SAVE_DIR: finetuned_fullpipeline_minilm_5ep
                     ALPHA_SEG: 0.4
           HYBRID_DENSE_WEIGHT: 0.72
           HYBRID_MORPH_WEIGHT: 0.28
          RERANK_HYBRID_WEIGHT: 0.75
             RERANK_ANS_WEIGHT: 0.25
                    DENSE_TOPK: 80
                   RERANK_TOPK: 12

✅ CSV exported: full_pipeline_test_details.csv  (rows=5028)
